# RateMyProfessor Tenure Impact Analysis

This notebook answers four research questions using individual RMP reviews
collected for 93 CS professors, each with a known (or estimated) tenure year.

**Before running this notebook**, make sure you have:
1. Ran the review crawler: `python src/review_crawler.py`
2. Installed dependencies: `pip install -r requirements.txt`

---

## Research Questions

| # | Question |
|---|----------|
| Q1 | Do **quality / difficulty / would-take-again** ratings change before vs. after tenure? |
| Q2 | What are the **semantic changes** (sentiment, keywords, tags) in reviews? |
| Q3 | Does **review frequency** (reviews per year) change after tenure? |
| Q4 | Does the **number of years** before/after tenure moderate those changes? |

In [ ]:
import sys, warnings
from pathlib import Path

# Allow importing project modules from the repo root
REPO_ROOT = Path("__file__").resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats as scipy_stats

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Download NLTK corpora if missing
for corpus in ('vader_lexicon', 'stopwords', 'punkt'):
    try:
        nltk.data.find(corpus)
    except LookupError:
        nltk.download(corpus, quiet=True)

print('Setup complete.')

In [ ]:
import config

REVIEWS_PATH   = config.REVIEWS_ALL_FILE
PROFS_PATH     = config.PROFESSORS_FILTERED_CSV
ANALYSIS_DIR   = config.ANALYSIS_DIR
PLOTS_DIR      = config.ANALYSIS_PLOTS_DIR
MIN_REVIEWS    = config.MIN_REVIEWS_PER_PERIOD   # default 3

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Reviews:    {REVIEWS_PATH}')
print(f'Professors: {PROFS_PATH}')
print(f'Output dir: {ANALYSIS_DIR}')

## Data Loading & Overview

In [ ]:
df_rev  = pd.read_csv(REVIEWS_PATH, low_memory=False)
df_prof = pd.read_csv(PROFS_PATH)

# Coerce types
for col in ['review_year', 'tenure_year', 'quality', 'difficulty', 'years_from_tenure']:
    df_rev[col] = pd.to_numeric(df_rev[col], errors='coerce')

def normalise_wta(v):
    if pd.isna(v): return np.nan
    s = str(v).lower().strip()
    if s in ('true','1','yes'): return 1.0
    if s in ('false','0','no'): return 0.0
    return np.nan
df_rev['would_take_again'] = df_rev['would_take_again'].apply(normalise_wta)

# Keep only labelled periods
df_rev = df_rev[df_rev['period'].isin(['pre_tenure','post_tenure'])].copy()

print(f'Total reviews: {len(df_rev):,}')
print(df_rev['period'].value_counts())

In [ ]:
# Professors with enough reviews in BOTH periods
counts = df_rev.groupby(['professor_slug','period']).size().unstack(fill_value=0)
both_mask = (counts.get('pre_tenure',0) >= MIN_REVIEWS) & (counts.get('post_tenure',0) >= MIN_REVIEWS)
valid_slugs = counts[both_mask].index
df_both = df_rev[df_rev['professor_slug'].isin(valid_slugs)].copy()

print(f'Professors with ≥{MIN_REVIEWS} reviews in BOTH periods: {len(valid_slugs)} / {df_rev["professor_slug"].nunique()}')
print(f'Reviews in filtered set: {len(df_both):,}')

# Overview table
overview = (
    df_both.groupby('professor_slug')
    .agg(
        professor_name=('professor_name','first'),
        university=('university','first'),
        tenure_year=('tenure_year','first'),
        n_pre=('period', lambda x: (x=='pre_tenure').sum()),
        n_post=('period', lambda x: (x=='post_tenure').sum()),
    )
    .reset_index()
)
display(overview.head(10))

In [ ]:
# Review timeline: how many reviews per year across all professors
yearly = df_rev.groupby(['review_year','period']).size().reset_index(name='count')
fig, ax = plt.subplots(figsize=(12,4))
for period, color, label in [('pre_tenure','steelblue','Pre-tenure'),('post_tenure','coral','Post-tenure')]:
    sub = yearly[yearly['period']==period]
    ax.bar(sub['review_year'], sub['count'], alpha=0.7, color=color, label=label)
ax.set_title('Number of RMP Reviews per Year Across All Professors')
ax.set_xlabel('Review Year')
ax.set_ylabel('Review Count')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'overview_timeline.png', bbox_inches='tight')
plt.show()

---
## Q1 — Do Quality / Difficulty / Would-Take-Again Change After Tenure?

**Method:** For each professor compute the mean of each metric in the pre- and
post-tenure periods.  Then run a **Wilcoxon signed-rank test** on the paired
means (non-parametric; robust to small or skewed samples).  Report Cohen's *d*
as an effect-size measure.

In [ ]:
METRICS = {
    'quality':         'Quality (avg helpful+clarity, 1–5)',
    'difficulty':      'Difficulty (1–5, higher = harder)',
    'would_take_again':'Would Take Again (0=No, 1=Yes)',
}

agg = (
    df_both
    .groupby(['professor_slug','professor_name','period'])[list(METRICS)]
    .mean()
    .reset_index()
)
pivot = agg.pivot_table(
    index=['professor_slug','professor_name'],
    columns='period',
    values=list(METRICS)
)
pivot.columns = [f'{m}_{p}' for m,p in pivot.columns]
pivot = pivot.reset_index()

for m in METRICS:
    pre_c  = f'{m}_pre_tenure'
    post_c = f'{m}_post_tenure'
    if pre_c in pivot.columns and post_c in pivot.columns:
        pivot[f'{m}_delta'] = pivot[post_c] - pivot[pre_c]

display(pivot.head(8))

In [ ]:
def cohens_d(a, b):
    diff = a - b
    return diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) != 0 else np.nan

rows = []
for m, label in METRICS.items():
    pre_c  = f'{m}_pre_tenure'
    post_c = f'{m}_post_tenure'
    if pre_c not in pivot or post_c not in pivot:
        continue
    valid = pivot[[pre_c, post_c, f'{m}_delta']].dropna()
    pre_arr  = valid[pre_c].values
    post_arr = valid[post_c].values
    stat, p  = scipy_stats.wilcoxon(pre_arr, post_arr, alternative='two-sided')
    d        = cohens_d(post_arr, pre_arr)
    rows.append(dict(
        metric=m, label=label,
        n=len(valid),
        mean_pre=round(float(pre_arr.mean()),3),
        mean_post=round(float(post_arr.mean()),3),
        mean_delta=round(float(valid[f'{m}_delta'].mean()),3),
        cohens_d=round(float(d),3),
        p_value=round(float(p),4),
        significant='✓' if p < 0.05 else '✗',
        direction='↑ higher post-tenure' if valid[f'{m}_delta'].mean()>0 else '↓ lower post-tenure'
    ))

q1_summary = pd.DataFrame(rows)
q1_summary.to_csv(ANALYSIS_DIR / 'q1_summary.csv', index=False)
pivot.to_csv(ANALYSIS_DIR / 'q1_numeric_changes.csv', index=False)
display(q1_summary[['metric','n','mean_pre','mean_post','mean_delta','cohens_d','p_value','significant','direction']])

In [ ]:
n = len(METRICS)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1: axes = [axes]

for ax, (m, label) in zip(axes, METRICS.items()):
    pre_c  = f'{m}_pre_tenure'
    post_c = f'{m}_post_tenure'
    if pre_c not in pivot or post_c not in pivot:
        continue
    valid = pivot[[pre_c, post_c]].dropna()
    long  = pd.DataFrame({'Pre-tenure': valid[pre_c].values, 'Post-tenure': valid[post_c].values})
    long_m = long.melt(var_name='Period', value_name='Value')
    sns.violinplot(data=long_m, x='Period', y='Value', ax=ax, inner='box', cut=0,
                   palette=['steelblue','coral'])
    for _, row in valid.iterrows():
        ax.plot([0,1],[row[pre_c], row[post_c]], color='grey', alpha=0.3, lw=0.8)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('')

fig.suptitle('Q1: Numeric Metric Changes Before vs. After Tenure\n(grey lines = individual professors)', fontsize=13)
fig.tight_layout()
plt.savefig(PLOTS_DIR/'q1_metric_violins.png', bbox_inches='tight')
plt.show()

In [ ]:
delta_cols = [f'{m}_delta' for m in METRICS if f'{m}_delta' in pivot.columns]
delta_data = pivot[delta_cols].dropna(how='all')
delta_data.columns = [c.replace('_delta','') for c in delta_data.columns]
delta_long = delta_data.melt(var_name='Metric', value_name='Delta (post − pre)')

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=delta_long, x='Metric', y='Delta (post − pre)', ax=ax, palette='muted')
ax.axhline(0, color='red', linestyle='--', lw=1.2, label='No change')
ax.set_title('Q1: Distribution of Metric Deltas Across Professors (Post − Pre Tenure)')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR/'q1_delta_boxplot.png', bbox_inches='tight')
plt.show()

---
## Q2 — Semantic Changes in Reviews Before vs. After Tenure

Three lenses:
- **(a) Sentiment** — VADER compound score per review, aggregated per professor
- **(b) Keywords** — TF-IDF top-50 terms for each corpus (pre / post)
- **(c) Tags** — frequency of RMP topic tags (e.g. "TOUGH GRADER", "CARING")

In [ ]:
sia = SentimentIntensityAnalyzer()
df_text = df_both[df_both['comment'].notna() & (df_both['comment'] != '')].copy()

df_text['sentiment_compound'] = df_text['comment'].apply(
    lambda t: sia.polarity_scores(str(t))['compound']
)
df_text['sentiment_pos'] = df_text['comment'].apply(
    lambda t: sia.polarity_scores(str(t))['pos']
)
df_text['sentiment_neg'] = df_text['comment'].apply(
    lambda t: sia.polarity_scores(str(t))['neg']
)

print(f'Reviews with text: {len(df_text):,}')
print(df_text.groupby('period')['sentiment_compound'].describe())

In [ ]:
# Per-professor mean sentiment, paired test
sent_agg = (
    df_text
    .groupby(['professor_slug','professor_name','period'])
    [['sentiment_compound','sentiment_pos','sentiment_neg']]
    .mean().reset_index()
)
sent_pivot = sent_agg.pivot_table(
    index=['professor_slug','professor_name'],
    columns='period',
    values=['sentiment_compound','sentiment_pos','sentiment_neg']
)
sent_pivot.columns = [f'{m}_{p}' for m,p in sent_pivot.columns]
sent_pivot = sent_pivot.reset_index()

valid_sent = sent_pivot[['sentiment_compound_pre_tenure','sentiment_compound_post_tenure']].dropna()
stat, p = scipy_stats.wilcoxon(
    valid_sent['sentiment_compound_pre_tenure'],
    valid_sent['sentiment_compound_post_tenure']
)
print(f'Sentiment (compound)  pre={valid_sent["sentiment_compound_pre_tenure"].mean():.3f}  '
      f'post={valid_sent["sentiment_compound_post_tenure"].mean():.3f}  '
      f'Wilcoxon p={p:.4f}')

sent_pivot['sentiment_compound_delta'] = (
    sent_pivot['sentiment_compound_post_tenure'] - sent_pivot['sentiment_compound_pre_tenure']
)
sent_pivot.to_csv(ANALYSIS_DIR / 'q2_sentiment_changes.csv', index=False)
display(sent_pivot.head(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribution
for period, color, label in [
    ('pre_tenure','steelblue','Pre-tenure'),
    ('post_tenure','coral','Post-tenure')
]:
    sub = df_text[df_text['period']==period]['sentiment_compound'].dropna()
    axes[0].hist(sub, bins=30, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('Review Sentiment Score Distribution')
axes[0].set_xlabel('VADER Compound Score (−1=most negative, +1=most positive)')
axes[0].legend()

# Box plot
sent_d = df_text[['period','sentiment_compound']].dropna()
sent_d['Period'] = sent_d['period'].map({'pre_tenure':'Pre-tenure','post_tenure':'Post-tenure'})
sns.boxplot(data=sent_d, x='Period', y='sentiment_compound', ax=axes[1],
            palette=['steelblue','coral'])
axes[1].set_title('Sentiment by Period (all reviews)')
axes[1].set_xlabel('')
axes[1].set_ylabel('VADER Compound Score')

fig.suptitle('Q2a: Sentiment Analysis', fontsize=13)
fig.tight_layout()
plt.savefig(PLOTS_DIR/'q2_sentiment.png', bbox_inches='tight')
plt.show()

In [ ]:
STOP = set(stopwords.words('english'))
STOP.update(['professor','class','course','just','really','also','lot','take','get','go',
             'one','good','bad','great','much','make','even','gave','give','made','know'])

pre_corpus  = df_text[df_text['period']=='pre_tenure']['comment'].dropna().tolist()
post_corpus = df_text[df_text['period']=='post_tenure']['comment'].dropna().tolist()

print(f'Pre-tenure comments: {len(pre_corpus):,}   Post-tenure: {len(post_corpus):,}')

vec = TfidfVectorizer(
    stop_words=list(STOP), max_features=300,
    ngram_range=(1,2), min_df=3, sublinear_tf=True
)
vec.fit(pre_corpus + post_corpus)
vocab = vec.get_feature_names_out()

def top_kw(corpus, n=50):
    mat    = vec.transform(corpus)
    scores = np.asarray(mat.mean(axis=0)).flatten()
    idx    = np.argsort(scores)[::-1][:n]
    return pd.DataFrame({'keyword': vocab[idx], 'tfidf_score': scores[idx]})

kw_pre  = top_kw(pre_corpus)
kw_post = top_kw(post_corpus)
kw_pre .to_csv(ANALYSIS_DIR/'q2_keywords_pre.csv',  index=False)
kw_post.to_csv(ANALYSIS_DIR/'q2_keywords_post.csv', index=False)

print('\nTop-10 Pre-tenure keywords:')
print(kw_pre.head(10).to_string(index=False))
print('\nTop-10 Post-tenure keywords:')
print(kw_post.head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, kw_df, title, color in [
    (axes[0], kw_pre.head(20),  'Pre-tenure Top-20 Keywords',  'steelblue'),
    (axes[1], kw_post.head(20), 'Post-tenure Top-20 Keywords', 'coral'),
]:
    ax.barh(kw_df['keyword'][::-1], kw_df['tfidf_score'][::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Mean TF-IDF Score')

fig.suptitle('Q2b: Most Characteristic Keywords Before vs. After Tenure', fontsize=13)
fig.tight_layout()
plt.savefig(PLOTS_DIR/'q2_keywords.png', bbox_inches='tight')
plt.show()

In [ ]:
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, corpus, title in [
        (axes[0], pre_corpus,  'Pre-tenure Reviews'),
        (axes[1], post_corpus, 'Post-tenure Reviews'),
    ]:
        text = ' '.join(corpus)
        wc   = WordCloud(width=800, height=400, stopwords=STOP,
                         background_color='white', max_words=150).generate(text)
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(title, fontsize=13)
    fig.suptitle('Q2b: Word Clouds', fontsize=14)
    fig.tight_layout()
    plt.savefig(PLOTS_DIR/'q2_wordclouds.png', bbox_inches='tight')
    plt.show()
except ImportError:
    print('wordcloud not installed — run: pip install wordcloud')

In [ ]:
# Tag frequency analysis
tag_rows = []
for period in ['pre_tenure','post_tenure']:
    sub = df_both[df_both['period']==period]['tags'].dropna()
    for cell in sub:
        for tag in str(cell).split('|'):
            tag = tag.strip()
            if tag:
                tag_rows.append({'period': period, 'tag': tag})

tag_df = pd.DataFrame(tag_rows)
n_pre  = (df_both['period']=='pre_tenure').sum()
n_post = (df_both['period']=='post_tenure').sum()

tag_counts = tag_df.groupby(['period','tag']).size().reset_index(name='count')
tag_pivot  = tag_counts.pivot_table(index='tag', columns='period', values='count', fill_value=0).reset_index()
if 'pre_tenure'  in tag_pivot.columns: tag_pivot['pre_pct']  = tag_pivot['pre_tenure']  / n_pre  * 100
if 'post_tenure' in tag_pivot.columns: tag_pivot['post_pct'] = tag_pivot['post_tenure'] / n_post * 100
tag_pivot = tag_pivot.sort_values('pre_pct', ascending=False)
tag_pivot.to_csv(ANALYSIS_DIR/'q2_tag_frequencies.csv', index=False)

print('Top-10 tags by pre-tenure frequency:')
display(tag_pivot.head(10))

In [ ]:
top_tags = tag_pivot.head(20)
fig, ax  = plt.subplots(figsize=(12, 7))
x = np.arange(len(top_tags))
w = 0.35
ax.barh(x - w/2, top_tags['pre_pct'],  w, label='Pre-tenure',  color='steelblue')
ax.barh(x + w/2, top_tags['post_pct'], w, label='Post-tenure', color='coral')
ax.set_yticks(x)
ax.set_yticklabels(top_tags['tag'])
ax.set_xlabel('% of reviews containing this tag')
ax.set_title('Q2c: Top-20 Review Tags — Pre vs. Post Tenure')
ax.legend()
fig.tight_layout()
plt.savefig(PLOTS_DIR/'q2_tags.png', bbox_inches='tight')
plt.show()

---
## Q3 — Does Review Frequency Change After Tenure?

**Metric:** Reviews per year in the pre-tenure window vs. the post-tenure window.

For each professor:
$$\text{rate} = \frac{\text{# reviews}}{\text{max year} - \text{min year}}$$
(if the span is 0 we fall back to raw count).

In [ ]:
def review_rate(sub):
    years = sub['review_year'].dropna()
    if len(years) < 2: return float(len(sub))
    span = years.max() - years.min()
    return float(len(sub) / span) if span > 0 else float(len(sub))

freq_rows = []
for slug, grp in df_both.groupby('professor_slug'):
    pre  = grp[grp['period']=='pre_tenure']
    post = grp[grp['period']=='post_tenure']
    rp   = review_rate(pre)
    rpo  = review_rate(post)
    pre_years  = pre['review_year'].dropna()
    post_years = post['review_year'].dropna()
    freq_rows.append(dict(
        professor_slug=slug,
        professor_name=grp['professor_name'].iloc[0],
        university=grp['university'].iloc[0],
        tenure_year=grp['tenure_year'].iloc[0],
        n_pre=len(pre), n_post=len(post),
        pre_span=float(pre_years.max()-pre_years.min()) if len(pre_years)>=2 else None,
        post_span=float(post_years.max()-post_years.min()) if len(post_years)>=2 else None,
        rate_pre=round(rp,3),
        rate_post=round(rpo,3),
        rate_delta=round(rpo-rp,3),
    ))

freq_df = pd.DataFrame(freq_rows)
freq_df.to_csv(ANALYSIS_DIR/'q3_frequency_changes.csv', index=False)

valid_freq = freq_df[['rate_pre','rate_post','rate_delta']].dropna()
stat, p = scipy_stats.wilcoxon(valid_freq['rate_pre'], valid_freq['rate_post'])

print(f"Review frequency  pre={valid_freq['rate_pre'].mean():.2f}  "
      f"post={valid_freq['rate_post'].mean():.2f}  "
      f"delta={valid_freq['rate_delta'].mean():.2f}  Wilcoxon p={p:.4f}")
display(freq_df.head(8))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Violin + paired lines
rate_long = pd.DataFrame({
    'Pre-tenure':  valid_freq['rate_pre'].values,
    'Post-tenure': valid_freq['rate_post'].values,
}).melt(var_name='Period', value_name='Reviews per year')
sns.violinplot(data=rate_long, x='Period', y='Reviews per year', ax=axes[0],
               inner='box', cut=0, palette=['steelblue','coral'])
for _, row in valid_freq.iterrows():
    axes[0].plot([0,1],[row['rate_pre'],row['rate_post']], color='grey', alpha=0.3, lw=0.8)
axes[0].set_title('Q3: Review Rate Before vs. After Tenure')

# Delta histogram
axes[1].hist(valid_freq['rate_delta'], bins=20, color='teal', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_title('Q3: Distribution of Δ Review Rate (Post − Pre)')
axes[1].set_xlabel('Δ Reviews per year')

fig.tight_layout()
plt.savefig(PLOTS_DIR/'q3_frequency.png', bbox_inches='tight')
plt.show()

In [ ]:
# Per-professor review timeline heat-map (top professors by review count)
top_profs = (
    df_both.groupby('professor_slug').size()
    .nlargest(20).index
)
hm_data = (
    df_both[df_both['professor_slug'].isin(top_profs)]
    .groupby(['professor_name','review_year'])
    .size()
    .unstack(fill_value=0)
)

# Mark tenure year with a dashed column
fig, ax = plt.subplots(figsize=(14, max(5, len(hm_data)*0.4)))
sns.heatmap(hm_data, ax=ax, cmap='YlOrRd', linewidths=0.5, linecolor='grey',
            cbar_kws={'label':'# reviews'})
ax.set_title('Q3: Reviews per Year per Professor (top 20 by review count)')
ax.set_xlabel('Review Year')
ax.set_ylabel('')
fig.tight_layout()
plt.savefig(PLOTS_DIR/'q3_heatmap.png', bbox_inches='tight')
plt.show()

---
## Q4 — Does the Number of Years Before/After Tenure Moderate the Changes?

Two predictors per professor:
- **years_pre**  = `tenure_year − min(review year in pre period)`
- **years_post** = `max(review year in post period) − tenure_year`

We use **Spearman correlations** (non-parametric) between each predictor and
each metric delta from Q1/Q3.

In [ ]:
years_rows = []
for slug, grp in df_both.groupby('professor_slug'):
    ty   = grp['tenure_year'].iloc[0]
    pre  = grp[grp['period']=='pre_tenure']['review_year'].dropna()
    post = grp[grp['period']=='post_tenure']['review_year'].dropna()
    if pd.isna(ty): continue
    yrs_pre  = float(ty) - float(pre.min())  if len(pre)  > 0 else None
    yrs_post = float(post.max()) - float(ty) if len(post) > 0 else None
    years_rows.append({'professor_slug': slug, 'years_pre': yrs_pre, 'years_post': yrs_post})

years_df = pd.DataFrame(years_rows)

# Merge Q1 deltas
delta_cols = [f'{m}_delta' for m in METRICS if f'{m}_delta' in pivot.columns]
merged = years_df.merge(pivot[['professor_slug']+delta_cols], on='professor_slug', how='left')

# Merge Q3 rate delta
merged = merged.merge(
    freq_df[['professor_slug','rate_delta']].rename(columns={'rate_delta':'review_rate_delta'}),
    on='professor_slug', how='left'
)
merged.to_csv(ANALYSIS_DIR/'q4_correlations_raw.csv', index=False)
display(merged.head(5))

In [ ]:
outcomes  = delta_cols + ['review_rate_delta']
outcomes  = [c for c in outcomes if c in merged.columns]

corr_rows = []
for pred in ['years_pre','years_post']:
    for out in outcomes:
        pair = merged[[pred, out]].dropna()
        if len(pair) < 5:
            corr, p = np.nan, np.nan
        else:
            corr, p = scipy_stats.spearmanr(pair[pred], pair[out])
        corr_rows.append(dict(
            predictor=pred, outcome=out, n=len(pair),
            spearman_r=round(float(corr),3),
            p_value=round(float(p),4) if not np.isnan(p) else None,
            significant='✓' if (not np.isnan(p) and p<0.05) else '✗'
        ))

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(ANALYSIS_DIR/'q4_correlations.csv', index=False)
display(corr_df)

In [ ]:
predictors = [p for p in ['years_pre','years_post'] if p in merged.columns]
n_rows = len(predictors)
n_cols = len(outcomes)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows), squeeze=False)

for r, pred in enumerate(predictors):
    for c, out in enumerate(outcomes):
        ax   = axes[r][c]
        pair = merged[[pred, out]].dropna()
        ax.scatter(pair[pred], pair[out], alpha=0.7, s=60)
        if len(pair) >= 3:
            slope, intercept, *_ = scipy_stats.linregress(pair[pred], pair[out])
            xr = np.linspace(pair[pred].min(), pair[pred].max(), 100)
            ax.plot(xr, slope*xr+intercept, color='red', lw=1.5, label='OLS fit')
        ax.axhline(0, color='grey', linestyle='--', lw=0.8)
        ax.set_xlabel(pred.replace('_',' '))
        ax.set_ylabel(out.replace('_',' '))
        # Annotate Spearman r
        row = corr_df[(corr_df['predictor']==pred)&(corr_df['outcome']==out)]
        if len(row):
            r_val = row.iloc[0]['spearman_r']
            p_val = row.iloc[0]['p_value']
            ax.set_title(f'ρ={r_val}, p={p_val}', fontsize=9)

fig.suptitle('Q4: Years Before/After Tenure vs. Metric Changes', fontsize=13, y=1.01)
fig.tight_layout()
plt.savefig(PLOTS_DIR/'q4_scatter.png', bbox_inches='tight')
plt.show()

---
## Summary of Findings

In [ ]:
print('=' * 65)
print('SUMMARY OF TENURE IMPACT ANALYSIS')
print('=' * 65)

print('\n── Q1: Numeric Metrics ──')
display(q1_summary[['metric','mean_pre','mean_post','mean_delta','p_value','significant','direction']])

print('\n── Q2: Sentiment ──')
v = sent_pivot[['sentiment_compound_pre_tenure','sentiment_compound_post_tenure']].dropna()
print(f"  Mean sentiment  pre={v['sentiment_compound_pre_tenure'].mean():.3f}  "
      f"post={v['sentiment_compound_post_tenure'].mean():.3f}")
print('  Top-3 pre-tenure keywords:', ', '.join(kw_pre['keyword'].head(3).tolist()))
print('  Top-3 post-tenure keywords:', ', '.join(kw_post['keyword'].head(3).tolist()))

print('\n── Q3: Review Frequency ──')
print(f"  Mean reviews/year  pre={valid_freq['rate_pre'].mean():.2f}  "
      f"post={valid_freq['rate_post'].mean():.2f}  delta={valid_freq['rate_delta'].mean():.2f}")

print('\n── Q4: Years Before/After Tenure Correlations ──')
sig = corr_df[corr_df['significant']=='✓']
if len(sig):
    for _, row in sig.iterrows():
        print(f"  {row['predictor']} → {row['outcome']}: ρ={row['spearman_r']}, p={row['p_value']}")
else:
    print('  No significant correlations found (p < 0.05)')

print(f'\nAll CSVs saved to: {ANALYSIS_DIR}')
print(f'All plots saved to: {PLOTS_DIR}')